# Model Checkpointer

> Retrospective optimal checkpoint selection system as described in: [tuning_playbook-checkpointer](https://github.com/google-research/tuning_playbook?tab=readme-ov-file#saving-checkpoints-and-retrospectively-selecting-the-best-checkpoint)


In [ ]:
#| default_exp utils.checkpointer

In [ ]:
#| hide
from nbdev.showdoc import * 

In [ ]:
#| export
import os
import heapq
import torch

In [ ]:
#| export
class BaseCheckpointer:
    def __init__(self, cfg):
        self.checkpoint_dir = cfg.get('checkpoint_dir', 'checkpoints')
        os.makedirs(self.checkpoint_dir, exist_ok=True)

    def save_to_disk(self, model_state, file_name, score):
        filepath = os.path.join(self.checkpoint_dir, file_name)
        # Here you would save the model_state to disk (e.g., using torch.save for PyTorch)
        print(f"Saving checkpoint: {filepath} with score: {score}")
        torch.save(model_state, filepath)  # Uncomment when using actual model state

In [ ]:
#| export
class RetrospectiveCheckpointer(BaseCheckpointer):
    def __init__(self, cfg, checkpoint_dir=None, n_best=5, mode='min'):
        super().__init__(cfg)
        self.n_best = n_best
        self.mode = mode # 'min' for loss, 'max' for accuracy
        self.best_checkpoints = [] # List of (score, filename)

    def save_if_best(self, current_score, model_state, step):
        # Adjust score for min-heap if we are maximizing
        comparison_score = current_score if self.mode == 'min' else -current_score
        
        filename = f"checkpoint_step_{step}.pt"
        filepath = os.path.join(self.checkpoint_dir, filename)

        if len(self.best_checkpoints) < self.n_best:
            self.save_to_disk(model_state, filepath)
            heapq.heappush(self.best_checkpoints, (-comparison_score, filename))
        
        elif -comparison_score > self.best_checkpoints[0][0]:
            # Remove the worst of the "best"
            worst_score, worst_filename = heapq.heappop(self.best_checkpoints)
            os.remove(os.path.join(self.checkpoint_dir, worst_filename))
            
            # Save the new one
            self.save_to_disk(model_state, filepath)
            heapq.heappush(self.best_checkpoints, (-comparison_score, filename))

    def get_best(self):
        # Returns the filename with the absolute best score
        return max(self.best_checkpoints, key=lambda x: x[0])[1]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()